In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)
block_size = 8
batch_size = 4

cpu


In [2]:
with open('wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
vocabulary_size = len(chars)

In [3]:
string_to_int = { ch:i for i,ch in enumerate(chars) }
int_to_string = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join([int_to_string[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
print(data[:100])

tensor([92, 49, 66, 63,  1, 45, 76, 73, 68, 63, 61, 78,  1, 36, 79, 78, 63, 72,
        60, 63, 76, 65,  1, 63, 31, 73, 73, 69,  1, 73, 64,  1, 33, 73, 76, 73,
        78, 66, 83,  1, 59, 72, 62,  1, 78, 66, 63,  1, 52, 67, 84, 59, 76, 62,
         1, 67, 72,  1, 44, 84,  0,  1,  1,  1,  1,  0, 49, 66, 67, 77,  1, 63,
        31, 73, 73, 69,  1, 67, 77,  1, 64, 73, 76,  1, 78, 66, 63,  1, 79, 77,
        63,  1, 73, 64,  1, 59, 72, 83, 73, 72])


In [4]:
n = int(0.8*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    print(ix)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

x, y = get_batch('train')
print('inputs:')
print(x.shape)
print(x)
print('targets')
print(y)

tensor([ 14564,  71947,  71782, 162174])
inputs:
torch.Size([4, 8])
tensor([[78, 70, 63,  1, 65, 67, 76, 70],
        [33, 15, 57,  0,  0, 49, 66, 63],
        [78, 63, 76, 72,  1, 79, 74, 73],
        [66, 63, 72,  1, 81, 63,  1, 66]])
targets
tensor([[70, 63,  1, 65, 67, 76, 70,  1],
        [15, 57,  0,  0, 49, 66, 63, 72],
        [63, 76, 72,  1, 79, 74, 73, 72],
        [63, 72,  1, 81, 63,  1, 66, 59]])


In [5]:
block_size = 8

x = train_data[:block_size]
y = train_data[1:block_size + 1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print("when input is", context, "target is", target)


when input is tensor([92]) target is tensor(49)
when input is tensor([92, 49]) target is tensor(66)
when input is tensor([92, 49, 66]) target is tensor(63)
when input is tensor([92, 49, 66, 63]) target is tensor(1)
when input is tensor([92, 49, 66, 63,  1]) target is tensor(45)
when input is tensor([92, 49, 66, 63,  1, 45]) target is tensor(76)
when input is tensor([92, 49, 66, 63,  1, 45, 76]) target is tensor(73)
when input is tensor([92, 49, 66, 63,  1, 45, 76, 73]) target is tensor(68)


In [8]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocabulary_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocabulary_size, vocabulary_size)

    def forward(self, index, targets=None):

        logits = self.token_embedding_table(index)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape

            logits = logits.reshape(B * T, C)
            targets = targets.reshape(B * T)

            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, index, max_new_tokens):

        for _ in range(max_new_tokens):

            logits, loss = self.forward(index)

            logits = logits[:, -1, :]

            probs = F.softmax(logits, dim=-1)

            index_next = torch.multinomial(probs, num_samples=1)

            index = torch.cat((index, index_next), dim=1)

        return index


model = BigramLanguageModel(vocabulary_size)
m = model.to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)

generated_chars = decode(
    m.generate(context, max_new_tokens=500)[0].tolist()
)


    
print(context)
print(target)
print(int_to_string[context.item()])
print(int_to_string[target.item()])


first_letter = generated_chars[1]
print(first_letter)

token = string_to_int[first_letter]
print(token)


print(f" Generated Chars.start {generated_chars}")




tensor([[0]])
tensor(68)


j
P
45
 Generated Chars.start 
PiP]C0AN“-"QDnJkqTsu$ov[),‘b,69?'PVoD3'FfN7#eRJ[aEF$E0aN"u-Sqp“VnJCT*8[(:A‘C[TVn+x,OO7.h6‘“:QSL6
Cf/pr#O0&/%(—﻿qe,&j#*PTjvG—﻿m_PRlrX$%x[9-1/_Pg”U5‘8O5](&p? IIII?XT]
AVnA“DIJCNAm?•SM“p#OgKRp?y,PiH*”$—fQ_(N
!$*d)K/%s1y—,OdAGLXj_7)XjOv(_v5D1
kYE2r401XoYBxHCGQ•1z"?!$6•8mcGr4™_v﻿z',"AI9/b6
x—md[x™]Z2lVMAIIJepioy?9D™a/
CK”’Y™1-_vLXjw)*XG/KK” %U•13(F™g™%ed!(6k#:d‘&﻿69)HnqbqmNNdlim!r—WS;’!]9itK$$aK4BA’Kdm&;Uv?y?""WEU2eSm“fS$cyu$—O;(4&0zzDX3$%PRnz—c2U0Pi,s1FU8f’YRs—Qvcp“mYx!$L[-)!"+$gx’beGU8i ”Tg"7Vk+sZX
